# **Hospital Readmission Rates - ML Pipeline**

## Objectives

* Create Modelling Dataset
* Push data through ML pipeline for predictive analytics

## Inputs

* Dataset from the "Clean Data" file, imported from Kaggle

## Outputs

* Modelling Data, stored in "Modelling Data" file
* ML Model
    * Trained
    * Tested
    * Evaluated




---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\naqas\\OneDrive\\Documents\\Coding\\CI_Projects\\hospital_readmission_rates_analysis\\hospital_readmission_rates_analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\naqas\\OneDrive\\Documents\\Coding\\CI_Projects\\hospital_readmission_rates_analysis\\hospital_readmission_rates_analysis'

---

# Import Packages

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from imblearn.over_sampling import SMOTE

---

# Import Clean Data 

In [5]:
clean_hospital_data_df = pd.read_csv("data_files/CleanData/clean_hospital_data.csv")
clean_hospital_data_df

,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),1,41,0,1,0,0,0,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),3,59,0,18,0,0,0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),2,11,5,13,2,0,1,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),2,44,1,16,0,0,0,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,51,0,8,0,0,0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99467,AfricanAmerican,Male,[70-80),3,51,0,16,0,0,0,...,No,Down,No,No,No,No,No,Ch,Yes,>30
99468,AfricanAmerican,Female,[80-90),5,33,3,18,0,0,1,...,No,Steady,No,No,No,No,No,No,Yes,NO
99469,Caucasian,Male,[70-80),1,53,0,9,1,0,0,...,No,Down,No,No,No,No,No,Ch,Yes,NO
99470,Caucasian,Female,[80-90),10,45,2,21,0,0,1,...,No,Up,No,No,No,No,No,Ch,Yes,NO


---

## _**Machine Learning Pipeline**_

### _Encoding_

All categories must be encoded before ML model training. The types of encoding used is as follows:

* Mapped encoding - mapping "1" and "0" to binary columns
* One-hot encoding - encoding of values with no natural order
* Ordinal encoding - encoding of values with natural order

As the target variable is "Readmitted", for the modelling we will be binarising it.

In [6]:
clean_hospital_data_df["readmitted"] = (clean_hospital_data_df["readmitted"] == "<30").astype(int)
clean_hospital_data_df["readmitted"].value_counts()

readmitted
0    88308
1    11164
Name: count, dtype: int64

We will also be binarising the two testing columns (Max Glucose Serum & A1Cresult)

In [7]:
clean_hospital_data_df["max_glu_serum"] = (clean_hospital_data_df["max_glu_serum"] != "No Test Performed").astype(int)
clean_hospital_data_df["max_glu_serum"].value_counts()

max_glu_serum
0    94183
1     5289
Name: count, dtype: int64

In [8]:
clean_hospital_data_df["A1Cresult"] = (clean_hospital_data_df["A1Cresult"] != "No Test Performed").astype(int)
clean_hospital_data_df["A1Cresult"].value_counts()

A1Cresult
0    82879
1    16593
Name: count, dtype: int64

Next, we will map encode all binarised categories.

In [9]:
clean_hospital_data_df["gender_encoded"] = clean_hospital_data_df["gender"].map({"Male":1, "Female":0})
clean_hospital_data_df["diabetesMed_encoded"] = clean_hospital_data_df["diabetesMed"].map({"Yes":1,"No":0})
clean_hospital_data_df["change_encoded"] = clean_hospital_data_df["change"].map({"Ch":1,"No":0})

clean_hospital_data_df

,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,gender_encoded,diabetesMed_encoded,change_encoded
0,Caucasian,Female,[0-10),1,41,0,1,0,0,0,...,No,No,No,No,No,No,0,0,0,0
1,Caucasian,Female,[10-20),3,59,0,18,0,0,0,...,No,No,No,No,Ch,Yes,0,0,1,1
2,AfricanAmerican,Female,[20-30),2,11,5,13,2,0,1,...,No,No,No,No,No,Yes,0,0,1,0
3,Caucasian,Male,[30-40),2,44,1,16,0,0,0,...,No,No,No,No,Ch,Yes,0,1,1,1
4,Caucasian,Male,[40-50),1,51,0,8,0,0,0,...,No,No,No,No,Ch,Yes,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99467,AfricanAmerican,Male,[70-80),3,51,0,16,0,0,0,...,No,No,No,No,Ch,Yes,0,1,1,1
99468,AfricanAmerican,Female,[80-90),5,33,3,18,0,0,1,...,No,No,No,No,No,Yes,0,0,1,0
99469,Caucasian,Male,[70-80),1,53,0,9,1,0,0,...,No,No,No,No,Ch,Yes,0,1,1,1
99470,Caucasian,Female,[80-90),10,45,2,21,0,0,1,...,No,No,No,No,Ch,Yes,0,0,1,1


Next, all columns requiring one-hot encoding will be encoded.

In [10]:
clean_hospital_data_df = pd.get_dummies(clean_hospital_data_df, columns=["race","diag_1", "diag_2","diag_3","metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", 
             "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", 
             "acarbose", "miglitol", "troglitazone", "tolazamide", "examide", "citoglipton", "insulin", 
             "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone", "metformin-rosiglitazone", 
             "metformin-pioglitazone"], drop_first=True)
clean_hospital_data_df.columns.to_list()

['gender',
 'age',
 'time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'max_glu_serum',
 'A1Cresult',
 'change',
 'diabetesMed',
 'readmitted',
 'gender_encoded',
 'diabetesMed_encoded',
 'change_encoded',
 'race_Asian',
 'race_Caucasian',
 'race_Hispanic',
 'race_Other',
 'diag_1_Diabetes',
 'diag_1_Digestive',
 'diag_1_Endocrine',
 'diag_1_Genitourinary',
 'diag_1_Injury',
 'diag_1_Musculoskeletal',
 'diag_1_Neoplasms',
 'diag_1_Other',
 'diag_1_Respiratory',
 'diag_2_Diabetes',
 'diag_2_Digestive',
 'diag_2_Endocrine',
 'diag_2_Genitourinary',
 'diag_2_Injury',
 'diag_2_Musculoskeletal',
 'diag_2_Neoplasms',
 'diag_2_No diagnosis',
 'diag_2_Other',
 'diag_2_Respiratory',
 'diag_3_Diabetes',
 'diag_3_Digestive',
 'diag_3_Endocrine',
 'diag_3_Genitourinary',
 'diag_3_Injury',
 'diag_3_Musculoskeletal',
 'diag_3_Neoplasms',
 'diag_3_No diagnosis',
 'diag_3_Other',
 'diag_3_Respiratory',
 'me

Finally, Ordinal encoding must be performed on the Age column as the values are set as ranges.

In [11]:
age_order = [['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)',
       '[60-70)', '[70-80)', '[80-90)', '[90-100)']]

oe = OrdinalEncoder(categories=age_order)
clean_hospital_data_df["age_encoded"] = oe.fit_transform(clean_hospital_data_df[["age"]])
clean_hospital_data_df.columns.to_list()

['gender',
 'age',
 'time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'max_glu_serum',
 'A1Cresult',
 'change',
 'diabetesMed',
 'readmitted',
 'gender_encoded',
 'diabetesMed_encoded',
 'change_encoded',
 'race_Asian',
 'race_Caucasian',
 'race_Hispanic',
 'race_Other',
 'diag_1_Diabetes',
 'diag_1_Digestive',
 'diag_1_Endocrine',
 'diag_1_Genitourinary',
 'diag_1_Injury',
 'diag_1_Musculoskeletal',
 'diag_1_Neoplasms',
 'diag_1_Other',
 'diag_1_Respiratory',
 'diag_2_Diabetes',
 'diag_2_Digestive',
 'diag_2_Endocrine',
 'diag_2_Genitourinary',
 'diag_2_Injury',
 'diag_2_Musculoskeletal',
 'diag_2_Neoplasms',
 'diag_2_No diagnosis',
 'diag_2_Other',
 'diag_2_Respiratory',
 'diag_3_Diabetes',
 'diag_3_Digestive',
 'diag_3_Endocrine',
 'diag_3_Genitourinary',
 'diag_3_Injury',
 'diag_3_Musculoskeletal',
 'diag_3_Neoplasms',
 'diag_3_No diagnosis',
 'diag_3_Other',
 'diag_3_Respiratory',
 'me

In [12]:
clean_hospital_data_df.drop(columns=["gender", "age", "change", "diabetesMed"], inplace=True)
clean_hospital_data_df.columns.to_list()

['time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'max_glu_serum',
 'A1Cresult',
 'readmitted',
 'gender_encoded',
 'diabetesMed_encoded',
 'change_encoded',
 'race_Asian',
 'race_Caucasian',
 'race_Hispanic',
 'race_Other',
 'diag_1_Diabetes',
 'diag_1_Digestive',
 'diag_1_Endocrine',
 'diag_1_Genitourinary',
 'diag_1_Injury',
 'diag_1_Musculoskeletal',
 'diag_1_Neoplasms',
 'diag_1_Other',
 'diag_1_Respiratory',
 'diag_2_Diabetes',
 'diag_2_Digestive',
 'diag_2_Endocrine',
 'diag_2_Genitourinary',
 'diag_2_Injury',
 'diag_2_Musculoskeletal',
 'diag_2_Neoplasms',
 'diag_2_No diagnosis',
 'diag_2_Other',
 'diag_2_Respiratory',
 'diag_3_Diabetes',
 'diag_3_Digestive',
 'diag_3_Endocrine',
 'diag_3_Genitourinary',
 'diag_3_Injury',
 'diag_3_Musculoskeletal',
 'diag_3_Neoplasms',
 'diag_3_No diagnosis',
 'diag_3_Other',
 'diag_3_Respiratory',
 'metformin_No',
 'metformin_Steady',
 'metformin_

---

### _Create new Dataframe from new dataset_

In [13]:
clean_hospital_data_df.to_csv("data_files/ModellingData/modelling_hospital_data.csv", index = False)

In [17]:
hospital_modelling_df = pd.read_csv("data_files/ModellingData/modelling_hospital_data.csv")
hospital_modelling_df

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,max_glu_serum,A1Cresult,readmitted,...,insulin_No,insulin_Steady,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-pioglitazone_Steady,age_encoded
0,1,41,0,1,0,0,0,0,0,0,...,True,False,False,True,False,False,False,False,False,0.0
1,3,59,0,18,0,0,0,0,0,0,...,False,False,True,True,False,False,False,False,False,1.0
2,2,11,5,13,2,0,1,0,0,0,...,True,False,False,True,False,False,False,False,False,2.0
3,2,44,1,16,0,0,0,0,0,0,...,False,False,True,True,False,False,False,False,False,3.0
4,1,51,0,8,0,0,0,0,0,0,...,False,True,False,True,False,False,False,False,False,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99467,3,51,0,16,0,0,0,0,1,0,...,False,False,False,True,False,False,False,False,False,7.0
99468,5,33,3,18,0,0,1,0,0,0,...,False,True,False,True,False,False,False,False,False,8.0
99469,1,53,0,9,1,0,0,0,0,0,...,False,False,False,True,False,False,False,False,False,7.0
99470,10,45,2,21,0,0,1,0,0,0,...,False,False,True,True,False,False,False,False,False,8.0


---

### _Define Features and Target Variables_

In [18]:
X = hospital_modelling_df.drop(["readmitted"], axis=1)
y = hospital_modelling_df["readmitted"]

The features and the targets for the models were defined.

---

### _Train-Test Split_

In [20]:
X_train,X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, stratify=y, random_state=38)

The data was split into an 80:20 train test split. The "Stratify" function was used to maintain data proportionality in the train test split.

---

### _Scaling_

In [21]:
diabetes_scaler = StandardScaler()
X_train = diabetes_scaler.fit_transform(X_train)
X_test = diabetes_scaler.transform(X_test)

Scaling is done to balance the weights of different numerical values within the modelling dataset.

---

### _Class Imbalance_

In [22]:
smote = SMOTE( random_state = 17)
X_train, y_train = smote.fit_resample(X_train, y_train)

This is health data and it is very common to have class imbalances in health data. To deal with the imbalance, SMOTE is used to create synthetic data and balance the classes. This will not affect the train-test split as it has been done prior to fixing the class imbalance.

---

### _Model Training_

In [24]:
lr_model = LogisticRegression( random_state= 44, max_iter= 1000)
rf_model = RandomForestClassifier(random_state=26)

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=26)

Logistic regression and Random forest models were used. The Logistic Regression model initially reached it's max iterations before being able to find the optimal solution therefore, the max iterations were manually adjusted to "1000" to allow the model to work until the optimal solution was found.

---

### _Model Evaluation_

It is necessary to evaluate the accurace of the models to help determine which model is best for predicion with this dataset.

In [25]:
for name, model in [("Logistic Regression", lr_model),("Random Forest", rf_model)]:
    y_prediction = model.predict(X_test)
    print(f"\n{name}")
    print(f"Accuracy:{accuracy_score(y_test,y_prediction):.2f}")
    print(classification_report(y_test, y_prediction))


Logistic Regression
Accuracy:0.63
              precision    recall  f1-score   support

           0       0.91      0.65      0.76     17662
           1       0.15      0.51      0.24      2233

    accuracy                           0.63     19895
   macro avg       0.53      0.58      0.50     19895
weighted avg       0.83      0.63      0.70     19895


Random Forest
Accuracy:0.89
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     17662
           1       0.40      0.02      0.03      2233

    accuracy                           0.89     19895
   macro avg       0.65      0.51      0.49     19895
weighted avg       0.83      0.89      0.84     19895

